# MIMIC-III Paper-Style Experiment Runner

This notebook runs the current MIMIC-III paper-style relational experiments with the rebuilt wide table, short grid outputs, and shared final summaries.

In [48]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.append(str(NOTEBOOK_DIR))


## Import and Reload the Backend Modules

This cell reloads the feature definitions and shared runner so notebook reruns always use the latest local code.

In [49]:
import importlib
from copy import deepcopy

import pandas as pd
from IPython.display import display

import variable_sets
import window_experiment_common

importlib.reload(variable_sets)
importlib.reload(window_experiment_common)

run_experiment = window_experiment_common.run_experiment
OUT_DIR = Path(window_experiment_common.OUT_DIR)
RUN_OUTPUT_DIR = Path(window_experiment_common.RUN_OUTPUT_DIR)
SUMMARY_OUTPUT_DIR = Path(window_experiment_common.SUMMARY_OUTPUT_DIR)
EXPLAIN_OUTPUT_DIR = Path(window_experiment_common.EXPLAIN_OUTPUT_DIR)

pd.set_option("display.max_columns", 200)


## Define the Shared Summary File

All grid runs append horizon-level metrics into one shared CSV so you can compare many configurations in the same notebook.

In [50]:
SUMMARY_TABLE_NAME = "mimiciii_notebook_summary.csv"
SUMMARY_PATH = SUMMARY_OUTPUT_DIR / SUMMARY_TABLE_NAME
SUMMARY_PATH


WindowsPath('d:/ESILV_S2/Intern/mimic_paper_reproduction/mimiciii_test/output/grid_summary/mimiciii_notebook_summary.csv')

## Build the 12 Requested Grid Configurations

This cell generates the six window choices crossed with the two split modes so the notebook can run the full batch sequentially.

In [51]:
WINDOW_SPECS = [
    {"label": "+6_-6", "cohort_before": 6, "cohort_after": 6},
    {"label": "+3_-6", "cohort_before": 6, "cohort_after": 3},
    {"label": "+0_-6", "cohort_before": 6, "cohort_after": 0},
    {"label": "+8_-8", "cohort_before": 8, "cohort_after": 8},
    {"label": "+3_-9", "cohort_before": 9, "cohort_after": 3},
    {"label": "+0_-9", "cohort_before": 9, "cohort_after": 0},
]

SPLIT_SPECS = [
    {"split_mode": "cv10", "split_tag": "cv10"},
    {"split_mode": "cv5_80_20", "split_tag": "cv5"},
]

BASE_GRID_OPTIONS = {
    "include_static_in_root": False,
    "export_explainability": False,
    "negative_alignment_mode": "paper",
    "save_relational_tables": False,
    "save_curve_png": False,
    "display_plots": False,
    "append_summary": True,
    "summary_table_name": SUMMARY_TABLE_NAME,
}

EXPERIMENT_GRID = []
for window in WINDOW_SPECS:
    for split in SPLIT_SPECS:
        cfg = deepcopy(BASE_GRID_OPTIONS)
        cfg.update(
            {
                "task_name": f"MIMIC-III grid | {window['label']} | {split['split_tag']}",
                "task_prefix": f"mimiciii_{window['label']}_{split['split_tag']}".replace("+", "p").replace("-", "m"),
                "cohort_before": window["cohort_before"],
                "cohort_after": window["cohort_after"],
                "split_mode": split["split_mode"],
                "window_label": window["label"],
            }
        )
        EXPERIMENT_GRID.append(cfg)

pd.DataFrame(EXPERIMENT_GRID)[["task_prefix", "window_label", "cohort_before", "cohort_after", "split_mode"]]


,task_prefix,window_label,cohort_before,cohort_after,split_mode
0,mimiciii_p6_m6_cv10,+6_-6,6,6,cv10
1,mimiciii_p6_m6_cv5,+6_-6,6,6,cv5_80_20
2,mimiciii_p3_m6_cv10,+3_-6,6,3,cv10
3,mimiciii_p3_m6_cv5,+3_-6,6,3,cv5_80_20
4,mimiciii_p0_m6_cv10,+0_-6,6,0,cv10
5,mimiciii_p0_m6_cv5,+0_-6,6,0,cv5_80_20
6,mimiciii_p8_m8_cv10,+8_-8,8,8,cv10
7,mimiciii_p8_m8_cv5,+8_-8,8,8,cv5_80_20
8,mimiciii_p3_m9_cv10,+3_-9,9,3,cv10
9,mimiciii_p3_m9_cv5,+3_-9,9,3,cv5_80_20


## Run All Grid Configurations Sequentially

Each configuration prints a short result summary after it finishes. The cumulative in-memory table is `grid_short_results`, where each row is one configuration and the `3h` and `6h` metrics are shown side by side.

In [52]:
import contextlib
import io

RUN_GRID = True  # Set to True to run all 12 configurations; False to skip running and just display the grid.
grid_short_results = []

if RUN_GRID:
    for idx, cfg in enumerate(EXPERIMENT_GRID, start=1):
        print(f"[{idx}/{len(EXPERIMENT_GRID)}] {cfg['task_name']}")
        run_cfg = {k: v for k, v in cfg.items() if k != 'window_label'}
        _buffer = io.StringIO()
        with contextlib.redirect_stdout(_buffer):
            result = run_experiment(**run_cfg)
        result_df = pd.DataFrame(result["results"]).sort_values("h")
        horizon_lookup = result_df.set_index("h")
        grid_short_results.append(
            {
                "task_prefix": cfg["task_prefix"],
                "window_label": cfg["window_label"],
                "split_mode": cfg["split_mode"],
                "auc_3h": horizon_lookup.loc[3, "auc_roc"] if 3 in horizon_lookup.index else pd.NA,
                "auc_6h": horizon_lookup.loc[6, "auc_roc"] if 6 in horizon_lookup.index else pd.NA,
                "auprc_3h": horizon_lookup.loc[3, "auprc"] if 3 in horizon_lookup.index else pd.NA,
                "auprc_6h": horizon_lookup.loc[6, "auprc"] if 6 in horizon_lookup.index else pd.NA,
                "total_mean_train_sec": result_df["mean_train_sec"].sum(),
                "metrics_path": result["metrics_path"],
            }
        )
        latest = pd.DataFrame(grid_short_results).tail(1)
        latest_display = latest[[
            "task_prefix",
            "window_label",
            "split_mode",
            "auc_3h",
            "auc_6h",
            "auprc_3h",
            "auprc_6h",
            "total_mean_train_sec",
        ]]
        display(latest_display)
        print(
            f"    3h AUC={latest.iloc[0]['auc_3h']:.4f} | 6h AUC={latest.iloc[0]['auc_6h']:.4f} | "
            f"3h AUPRC={latest.iloc[0]['auprc_3h']:.4f} | 6h AUPRC={latest.iloc[0]['auprc_6h']:.4f}"
        )
else:
    print("Set RUN_GRID = True when you are ready to run all 12 configurations.")


[1/12] MIMIC-III grid | +6_-6 | cv10


,task_prefix,window_label,split_mode,auc_3h,auc_6h,auprc_3h,auprc_6h,total_mean_train_sec
0,mimiciii_p6_m6_cv10,+6_-6,cv10,0.978262,0.974492,0.98508,0.982903,20.135361


    3h AUC=0.9783 | 6h AUC=0.9745 | 3h AUPRC=0.9851 | 6h AUPRC=0.9829
[2/12] MIMIC-III grid | +6_-6 | cv5


,task_prefix,window_label,split_mode,auc_3h,auc_6h,auprc_3h,auprc_6h,total_mean_train_sec
1,mimiciii_p6_m6_cv5,+6_-6,cv5_80_20,0.97884,0.974996,0.985302,0.983355,18.225537


    3h AUC=0.9788 | 6h AUC=0.9750 | 3h AUPRC=0.9853 | 6h AUPRC=0.9834
[3/12] MIMIC-III grid | +3_-6 | cv10


...

    3h AUC=0.9870 | 6h AUC=0.9833 | 3h AUPRC=0.9909 | 6h AUPRC=0.9889
[11/12] MIMIC-III grid | +0_-9 | cv10


,task_prefix,window_label,split_mode,auc_3h,auc_6h,auprc_3h,auprc_6h,total_mean_train_sec
10,mimiciii_p0_m9_cv10,+0_-9,cv10,0.987363,0.983294,0.991104,0.988724,28.076574


    3h AUC=0.9874 | 6h AUC=0.9833 | 3h AUPRC=0.9911 | 6h AUPRC=0.9887
[12/12] MIMIC-III grid | +0_-9 | cv5


,task_prefix,window_label,split_mode,auc_3h,auc_6h,auprc_3h,auprc_6h,total_mean_train_sec
11,mimiciii_p0_m9_cv5,+0_-9,cv5_80_20,0.987377,0.9845,0.991194,0.989482,25.23568


    3h AUC=0.9874 | 6h AUC=0.9845 | 3h AUPRC=0.9912 | 6h AUPRC=0.9895


## Load the Shared Summary Table

This cell reads the accumulated horizon-level table into `summary_df`. It is the longest-form result table: each row represents one configuration at one prediction horizon, so `3h` and `6h` appear as separate rows identified by `horizon_h`.

In [56]:
if SUMMARY_PATH.exists():
    summary_df = pd.read_csv(SUMMARY_PATH).sort_values(["task_prefix", "horizon_h"]).reset_index(drop=True)
    display(summary_df)
else:
    summary_df = pd.DataFrame()
    print(f"Summary file not found: {SUMMARY_PATH}")


,task_name,task_prefix,cohort_before,cohort_after,horizon_h,split_mode,include_static_in_root,negative_alignment_mode,export_explainability,n_pos,n_neg,visible_rows_per_stay,hourly_rows,oof_auc_roc,oof_auprc,mean_fold_auc_roc,mean_fold_auprc,mean_train_sec
0,MIMIC-III grid | +0_-6 | cv10,mimiciii_p0_m6_cv10,6,0,3,cv10,False,paper,False,5371,5371,4,42968,0.978670,0.985394,0.978766,0.985477,11.593758
1,MIMIC-III grid | +0_-6 | cv10,mimiciii_p0_m6_cv10,6,0,6,cv10,False,paper,False,5371,5371,1,10742,0.974736,0.983266,0.974863,0.983282,8.432244
2,MIMIC-III grid | +0_-6 | cv5,mimiciii_p0_m6_cv5,6,0,3,cv5_80_20,False,paper,False,5371,5371,4,42968,0.978945,0.985337,0.979069,0.985408,11.613607
3,MIMIC-III grid | +0_-6 | cv5,mimiciii_p0_m6_cv5,6,0,6,cv5_80_20,False,paper,False,5371,5371,1,10742,0.975254,0.983315,0.975300,0.983356,7.949555
4,MIMIC-III grid | +0_-9 | cv10,mimiciii_p0_m9_cv10,9,0,3,cv10,False,paper,False,4420,4420,7,61880,0.987363,0.991104,0.987411,0.991149,15.414976
5,MIMIC-III grid | +0_-9 | cv10,mimiciii_p0_m9_cv10,9,0,6,cv10,False,paper,False,4420,4420,4,35360,0.983294,0.988724,0.983443,0.988829,12.661599
6,MIMIC-III grid | +0_-9 | cv5,mimiciii_p0_m9_cv5,9,0,3,cv5_80_20,False,paper,False,4420,4420,7,61880,0.987377,0.991194,0.987469,0.991239,14.159947
7,MIMIC-III grid | +0_-9 | cv5,mimiciii_p0_m9_cv5,9,0,6,cv5_80_20,False,paper,False,4420,4420,4,35360,0.984500,0.989482,0.984548,0.989528,11.075734
8,MIMIC-III grid | +3_-6 | cv10,mimiciii_p3_m6_cv10,6,3,3,cv10,False,paper,False,5370,5370,4,42960,0.978818,0.985470,0.978869,0.985513,11.589446
9,MIMIC-III grid | +3_-6 | cv10,mimiciii_p3_m6_cv10,6,3,6,cv10,False,paper,False,5370,5370,1,10740,0.975472,0.983534,0.975656,0.983596,8.628083


## Build the Internal Configuration Table

This cell builds `compact_summary_df` as an internal one-row-per-configuration table. It is used by the ranking and final reporting cells, so it no longer needs a separate full display.

In [57]:
if summary_df.empty:
    compact_summary_df = pd.DataFrame()
    print("Summary table is empty. Run the grid first.")
else:
    config_base_df = summary_df[[
        "task_prefix",
        "task_name",
        "cohort_before",
        "cohort_after",
        "split_mode",
        "n_pos",
        "n_neg",
        "negative_alignment_mode",
        "include_static_in_root",
        "export_explainability",
    ]].drop_duplicates(subset=["task_prefix"])
    config_base_df["window_label"] = (
        "+" + config_base_df["cohort_before"].astype(int).astype(str)
        + "_-" + config_base_df["cohort_after"].astype(int).astype(str)
    )

    horizon_metrics_df = (
        summary_df[["task_prefix", "horizon_h", "oof_auc_roc", "oof_auprc", "mean_train_sec"]]
        .pivot(index="task_prefix", columns="horizon_h", values=["oof_auc_roc", "oof_auprc", "mean_train_sec"])
        .sort_index(axis=1)
    )
    horizon_metrics_df.columns = [
        f"{metric.replace('oof_', '')}_{int(h)}h" for metric, h in horizon_metrics_df.columns
    ]
    horizon_metrics_df = horizon_metrics_df.reset_index()

    compact_summary_df = config_base_df.merge(horizon_metrics_df, on="task_prefix", how="left")
    compact_summary_df["auc_roc_delta_6h_minus_3h"] = compact_summary_df["auc_roc_6h"] - compact_summary_df["auc_roc_3h"]
    compact_summary_df["auprc_delta_6h_minus_3h"] = compact_summary_df["auprc_6h"] - compact_summary_df["auprc_3h"]
    compact_summary_df["total_mean_train_sec"] = compact_summary_df[["mean_train_sec_3h", "mean_train_sec_6h"]].sum(axis=1)
    compact_summary_df = compact_summary_df.sort_values(["auc_roc_3h", "auc_roc_6h", "auprc_3h", "auprc_6h"], ascending=False).reset_index(drop=True)
    print(f"Built compact_summary_df with {len(compact_summary_df)} configurations.")


Built compact_summary_df with 12 configurations.


## Rank the Best Configurations

This ranking view keeps `3h` and `6h` separate. Each configuration receives independent rank columns for AUC and AUPRC at both horizons.

In [58]:
TOP_K = 12

if compact_summary_df.empty:
    print("Compact summary is empty. Run the grid first.")
else:
    ranking_df = compact_summary_df[[
        "task_prefix",
        "window_label",
        "split_mode",
        "n_pos",
        "n_neg",
        "auc_roc_3h",
        "auc_roc_6h",
        "auprc_3h",
        "auprc_6h",
        "auc_roc_delta_6h_minus_3h",
        "auprc_delta_6h_minus_3h",
        "total_mean_train_sec",
    ]].copy()
    ranking_df["rank_auc_3h"] = ranking_df["auc_roc_3h"].rank(method="min", ascending=False).astype("Int64")
    ranking_df["rank_auc_6h"] = ranking_df["auc_roc_6h"].rank(method="min", ascending=False).astype("Int64")
    ranking_df["rank_auprc_3h"] = ranking_df["auprc_3h"].rank(method="min", ascending=False).astype("Int64")
    ranking_df["rank_auprc_6h"] = ranking_df["auprc_6h"].rank(method="min", ascending=False).astype("Int64")
    ranking_df = ranking_df.sort_values(["rank_auc_3h", "rank_auc_6h", "rank_auprc_3h", "rank_auprc_6h"]).head(TOP_K)
    display(ranking_df)


,task_prefix,window_label,split_mode,n_pos,n_neg,auc_roc_3h,auc_roc_6h,auprc_3h,auprc_6h,auc_roc_delta_6h_minus_3h,auprc_delta_6h_minus_3h,total_mean_train_sec,rank_auc_3h,rank_auc_6h,rank_auprc_3h,rank_auprc_6h
0,mimiciii_p0_m9_cv5,+9_-0,cv5_80_20,4420,4420,0.987377,0.984500,0.991194,0.989482,-0.002877,-0.001712,25.235680,1,1,1,1
1,mimiciii_p0_m9_cv10,+9_-0,cv10,4420,4420,0.987363,0.983294,0.991104,0.988724,-0.004070,-0.002380,28.076574,2,3,2,3
2,mimiciii_p3_m9_cv10,+9_-3,cv10,4419,4419,0.987128,0.983105,0.991033,0.988672,-0.004022,-0.002361,22.186530,3,4,3,4
3,mimiciii_p3_m9_cv5,+9_-3,cv5_80_20,4419,4419,0.986970,0.983317,0.990934,0.988861,-0.003653,-0.002073,20.432107,4,2,4,2
4,mimiciii_p8_m8_cv10,+8_-8,cv10,4643,4643,0.984319,0.981796,0.989061,0.987582,-0.002523,-0.001479,22.039004,5,6,6,6
5,mimiciii_p8_m8_cv5,+8_-8,cv5_80_20,4643,4643,0.984309,0.982377,0.989111,0.987937,-0.001932,-0.001175,19.352881,6,5,5,5
6,mimiciii_p0_m6_cv5,+6_-0,cv5_80_20,5371,5371,0.978945,0.975254,0.985337,0.983315,-0.003690,-0.002022,19.563162,7,8,9,10
7,mimiciii_p6_m6_cv5,+6_-6,cv5_80_20,5347,5347,0.978840,0.974996,0.985302,0.983355,-0.003844,-0.001947,18.225537,8,10,10,9
8,mimiciii_p3_m6_cv10,+6_-3,cv10,5370,5370,0.978818,0.975472,0.985470,0.983534,-0.003346,-0.001936,20.217529,9,7,7,7
9,mimiciii_p0_m6_cv10,+6_-0,cv10,5371,5371,0.978670,0.974736,0.985394,0.983266,-0.003934,-0.002128,20.026002,10,11,8,11
